## User-level database

The user-level database combines AI usage measures from the chat-level database with students' academic performance and Kahoot activity.

The academic performance and Kahoot data come from a separate student-level database. Student names are used only to connect these records to the anonymous student IDs and are removed immediately after the merge. 

In [1]:
import pandas as pd
import numpy as np

chat = pd.read_excel('databases/chat_level_final.xlsx')

roster = pd.read_excel('OneDrive/Névsor2025_20250702.xlsx')
id_map = pd.read_excel('hallgatoi_id.xlsx')

### Academic performance and Kahoot activity

The student-level data contain Kahoot participation and test scores. Variable names are standardized before merging them with the student IDs.

In [2]:
user = roster[
    ['Név',
     'Kahoot1', 'Kahoot2', 'Kahoot3', 'Kahoot4',
     'Kahoot5', 'Kahoot6', 'Kahoot7', 'Kahoot 8',
     'Kahoot9', 'Kahoot10', 'Kahoot11', 'Kahoot12',
     'Kahhot13', 'Kahoot_SUM',
     'ZH1_szazalek', 'ZH2_szazalek', 'ZH3_szazalek',
     'ZH4_szazalek', 'ZH1234']
]

user = user.rename(columns={
    'Név': 'name',
    'Kahoot 8': 'Kahoot8',
    'Kahhot13': 'Kahoot13',
    'ZH1_szazalek': 'T1',
    'ZH2_szazalek': 'T2',
    'ZH3_szazalek': 'T3',
    'ZH4_szazalek': 'T4',
    'ZH1234': 'T1234'
})

user = user.merge(id_map, left_on='name', right_on='nev', how='left')

user = user.drop(columns=['name', 'nev'])

cols = ['id'] + [col for col in user.columns if col != 'id']
user = user[cols]

## Kahoot activity

Kahoot participation was aggregated into three periods: before the incentives, during the incentivized period, and after the incentives.

Relative activity measures were calculated for each period. Two additional measures combine the incentivized and post-incentive periods and capture overall participation across all Kahoot sessions. These combined measures are used in the regression analysis.

In [3]:
user['act_pre'] = user[['Kahoot1', 'Kahoot2', 'Kahoot3']].sum(axis=1)

user['act_rew'] = user[
    ['Kahoot4', 'Kahoot5', 'Kahoot6', 'Kahoot7',
     'Kahoot8', 'Kahoot9', 'Kahoot10', 'Kahoot11']
].sum(axis=1)

user['act_post'] = user[['Kahoot12', 'Kahoot13']].sum(axis=1)

user['rel_act_pre'] = user['act_pre'] / 3
user['rel_act_rew'] = user['act_rew'] / 8
user['rel_act_post'] = user['act_post'] / 2

# Combined activity measures used in the regression analysis
user['rel_act_prerew'] = (
    user['act_pre'] + user['act_rew']
) / 11

user['rel_act_rewpost'] = (
    user['act_rew'] + user['act_post']
) / 10

user['rel_act'] = user['Kahoot_SUM'] / 13

print(
    user[
        ['act_pre', 'act_rew', 'act_post',
         'rel_act_pre', 'rel_act_rew', 'rel_act_post',
         'rel_act_rewpost', 'rel_act']
    ].head()
)

   act_pre  act_rew  act_post  rel_act_pre  rel_act_rew  rel_act_post  \
0        3        6         2          1.0        0.750           1.0   
1        3        1         0          1.0        0.125           0.0   
2        3        7         1          1.0        0.875           0.5   
3        3        6         1          1.0        0.750           0.5   
4        3        8         2          1.0        1.000           1.0   

   rel_act_rewpost   rel_act  
0              0.8  0.846154  
1              0.1  0.307692  
2              0.8  0.846154  
3              0.7  0.769231  
4              1.0  1.000000  


In [4]:
print(f'Rows: {len(user)}')
print(f'Unique students: {user["id"].nunique()}')
user.info()

Rows: 140
Unique students: 140
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 29 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               140 non-null    object 
 1   Kahoot1          140 non-null    int64  
 2   Kahoot2          140 non-null    int64  
 3   Kahoot3          140 non-null    int64  
 4   Kahoot4          140 non-null    int64  
 5   Kahoot5          140 non-null    int64  
 6   Kahoot6          140 non-null    int64  
 7   Kahoot7          140 non-null    int64  
 8   Kahoot8          140 non-null    int64  
 9   Kahoot9          140 non-null    int64  
 10  Kahoot10         140 non-null    int64  
 11  Kahoot11         140 non-null    int64  
 12  Kahoot12         140 non-null    int64  
 13  Kahoot13         140 non-null    int64  
 14  Kahoot_SUM       140 non-null    int64  
 15  T1               140 non-null    float64
 16  T2               140 non-null  

## Combined test score

The second and third test scores were combined into a single score. When only one of the two scores was available, the available score was retained.

In [5]:
user['T23'] = np.where(
    (user['T2'] > 0) & (user['T3'] > 0),
    (user['T2'] + user['T3']) / 2,
    np.where(
        user['T2'] > 0,
        user['T2'],
        user['T3']
    )
)

print(user['T23'].describe())

count    140.000000
mean       0.530755
std        0.278610
min        0.000000
25%        0.417981
50%        0.572183
75%        0.726810
max        0.951699
Name: T23, dtype: float64


## AI use

An indicator variable was created to distinguish students who used the AI tutor from those who did not.

Students were classified as AI users if their student ID was present in the chat-level database.

In [6]:
user['AI'] = user['id'].isin(chat['id']).astype(int)

print(user['AI'].value_counts())

AI
0    96
1    44
Name: count, dtype: int64


## Merging AI usage measures

User-level measures derived from the chat-level database are merged into the student-level database using the anonymous student ID.

In [7]:
chat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 990 entries, 0 to 989
Data columns (total 52 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   row_id               990 non-null    int64         
 1   chat_id              990 non-null    int64         
 2   question_id          990 non-null    int64         
 3   question             990 non-null    object        
 4   answer               990 non-null    object        
 5   q_time               990 non-null    datetime64[ns]
 6   a_time               990 non-null    object        
 7   id                   990 non-null    object        
 8   T1                   990 non-null    float64       
 9   T2                   990 non-null    float64       
 10  T3                   990 non-null    float64       
 11  T4                   990 non-null    float64       
 12  T1234                990 non-null    float64       
 13  question_length      990 non-null  

In [8]:
chat_vars = [
    'id',
    'profile',
    'total_questions',
    'rewarded_count',
    'non_rewarded_count',
    'rewarded_share',
    'non_rewarded_share',
    'chat_count',
    'avg_question_length',
    'avg_q_len_rew',
    'avg_q_len_post',
    'avg_answer_length',
    'avg_a_len_rew',
    'avg_a_len_post',
    'relevance_rate',
    'avg_rel_rew',
    'avg_rel_post',
    'hw_share',
    'avg_hw_rw',
    'avg_hw_nrw',
    'active_days',
    'active_days_rew',
    'active_days_post',
    'exam_panic_rate'
]

chat_user = (
    chat[chat_vars]
    .drop_duplicates(subset='id')
)

user = user.merge(
    chat_user,
    on='id',
    how='left'
)

print(f'Rows: {len(user)}')
print(f'Unique students: {user["id"].nunique()}')

Rows: 140
Unique students: 140


In [9]:
user.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 54 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   140 non-null    object 
 1   Kahoot1              140 non-null    int64  
 2   Kahoot2              140 non-null    int64  
 3   Kahoot3              140 non-null    int64  
 4   Kahoot4              140 non-null    int64  
 5   Kahoot5              140 non-null    int64  
 6   Kahoot6              140 non-null    int64  
 7   Kahoot7              140 non-null    int64  
 8   Kahoot8              140 non-null    int64  
 9   Kahoot9              140 non-null    int64  
 10  Kahoot10             140 non-null    int64  
 11  Kahoot11             140 non-null    int64  
 12  Kahoot12             140 non-null    int64  
 13  Kahoot13             140 non-null    int64  
 14  Kahoot_SUM           140 non-null    int64  
 15  T1                   140 non-null    flo

## Filling missing values

Students who did not use the AI tutor have no chat-level measures. Their profile is therefore classified as `non-user` before filling the remaining missing values with zero.

For chat-level measures, a value of zero indicates that the corresponding activity or measure was not observed.

In [10]:
user['profile'] = user['profile'].fillna('non-user')

user = user.fillna(0)

In [11]:
kahoot_vars = [
    'Kahoot1', 'Kahoot2', 'Kahoot3', 'Kahoot4',
    'Kahoot5', 'Kahoot6', 'Kahoot7', 'Kahoot8',
    'Kahoot9', 'Kahoot10', 'Kahoot11', 'Kahoot12',
    'Kahoot13', 'Kahoot_SUM'
]

user = user.drop(columns=kahoot_vars)

print(f'Rows: {len(user)}')
print(f'Unique students: {user["id"].nunique()}')
print(user['profile'].value_counts())
user.info()

Rows: 140
Unique students: 140
profile
non-user      96
occasional    31
regular       13
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 40 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   140 non-null    object 
 1   T1                   140 non-null    float64
 2   T2                   140 non-null    float64
 3   T3                   140 non-null    float64
 4   T4                   140 non-null    float64
 5   T1234                140 non-null    float64
 6   act_pre              140 non-null    int64  
 7   act_rew              140 non-null    int64  
 8   act_post             140 non-null    int64  
 9   rel_act_pre          140 non-null    float64
 10  rel_act_rew          140 non-null    float64
 11  rel_act_post         140 non-null    float64
 12  rel_act_prerew       140 non-null    float64
 13  rel_act_rewpost      140

In [12]:
user.to_excel('userlevel.xlsx', index=False)